In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CMACT10", con=connection2)

connection2.close()




In [2]:
base2

,actcod,actdes,actdescor,estregcod,actotorcita,actareintflg,actareintcod
0,A0,ATENCION CEN.DE REHAB.INTEG.DE PAC.CRONICO (CR...,ATENCION CRIPC,1,0,NaN,None
1,A2,ATENCION CENTRO OBSTETRICO,ATEN.CENT.OBST.,1,0,1.0,06
2,A3,ATENCION UNIDAD DE CUIDADOS INTENSIVOS,ATENCION UCI,1,0,NaN,None
3,A4,ATENCION UNIDAD DE CUIDADOS INTERMEDIOS,ATENCION UCIN,1,0,NaN,None
4,A5,ATENCION UNIDAD DE VIGILANCIA INTENSIVA,ATENCION UVI,1,0,NaN,None
5,A7,ATENCION HOSPITALIZACION AMBULATORIA,ATEN.HOSP.AMBU.,1,1,NaN,None
6,A8,ATENCION UNIDAD DE CUIDADOS CORONARIOS,ATEN.UNID.C.COR,1,0,NaN,None
7,B1,ATENCION NO MEDICA AMBULATORIA,ATE.NO MED.AMB.,1,1,NaN,None
8,B2,ACTIVIDADES ESPECIFICAS DE LOS PROFES. NO MEDICOS,AC.ES.P.NO MED.,0,0,NaN,None
9,B3,ACTIVIDADES ASISTENCIALES DE TECNICOS,ACT.ASI.TECNIC.,0,0,NaN,None


In [3]:
base2.columns

Index(['actcod', 'actdes', 'actdescor', 'estregcod', 'actotorcita',
       'actareintflg', 'actareintcod'],
      dtype='object')

In [4]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cmact10 ()')
base2.to_sql(name='tmp_cmact10', con=engine3, if_exists='replace', index=False)
#df.to_sql(name='temp_table', con=engine, if_exists='replace', index=False, method='multi', chunksize=1000, temporary=True)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmaho10 FALSO CON LO OBTENIDO DEL ORACLE


query="""
ALTER TABLE tmp_cmact10 
ALTER COLUMN actcod         TYPE character(2),
ALTER COLUMN actdes         TYPE character(50),
ALTER COLUMN actdescor      TYPE character(15),
ALTER COLUMN estregcod      TYPE character(1),
ALTER COLUMN actotorcita    TYPE character(2),
ALTER COLUMN actareintflg   TYPE numeric(1,0),
ALTER COLUMN actareintcod   TYPE character(2);



UPDATE cmact10 
SET 
actdes      = t.actdes, 
actdescor       = t.actdescor, 
estregcod       = t.estregcod, 
actotorcita     = t.actotorcita,
actareintflg    = t.actareintflg, 
actareintcod    = t.actareintcod

FROM tmp_cmact10 t 
WHERE cmact10.actcod = t.actcod;

INSERT INTO cmact10 (actcod,actdes, actdescor, estregcod, actotorcita,actareintflg, actareintcod) 
SELECT actcod,actdes, actdescor, estregcod, actotorcita,actareintflg, actareintcod
FROM tmp_cmact10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmact10 
    WHERE cmact10.actcod = tmp_cmact10.actcod
);


"""

c1= text(query)
connection3.execute(c1)




#BORRAMOS LAS TABLAS
base2 = pd.read_sql_query(f"SELECT * FROM cmact10", con=connection3)
query2="""
DROP TABLE tmp_cmact10;
"""

c2= text(query2)
connection3.execute(c2)
connection3.close()


In [5]:

#################################################################################################################################################################################################################################################################################

#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cmact10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query="""
UPDATE dim_activi 
SET des_act      = t.actdes, 
dco_act       = t.actdescor

FROM tmp_cmact10 t 
WHERE dim_activi.cod_act = t.actcod;

INSERT INTO dim_activi (cod_act, des_act, dco_act) 
SELECT actcod,actdes, actdescor
FROM tmp_cmact10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_activi 
    WHERE dim_activi.cod_act = tmp_cmact10.actcod
);

DROP TABLE tmp_cmact10;
"""
c1= text(query)
connection4.execute(c1)
connection4.close()
